# Phase 1 — Fish Identification & Extraction

Runs `src/fish_extractor/` (Grounded SAM 2: zero-shot Grounding DINO detection + SAM 2.1 segmentation) against the Phase 0 dataset (`data/raw_images/`, 64 species, ~1,460 images).

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

**What this notebook does, and why it's structured this way:**
1. Mounts Google Drive and clones the project repo **directly onto Drive**, not Colab's ephemeral local disk. This is the key design choice: it means the resumable per-image state, the extracted images, and the review page (with its image links) all survive a Colab disconnect/restart, and stay openable from Drive afterwards - not just for the duration of one runtime.
2. Installs the GPU-only dependencies on top of Colab's preinstalled `torch`/CUDA (installing the project's `vision` extra as-is would try to reinstall those and risk a version mismatch, so it's done selectively here).
3. Runs the pipeline. It's resumable per-image (state is saved to disk after every single image), so a mid-run disconnect just means re-running the same cell - already-processed images are skipped automatically.
4. Generates the human-review page for anything the QA gate couldn't auto-accept, and applies your review decisions back.

See [README.md](../README.md) (Planned Approach, step 1) and [CLAUDE.md](../CLAUDE.md) for the full pipeline design and QA-gate reasoning.

## 0. Confirm you actually have a GPU attached

Do this *before* anything else - `Runtime -> Change runtime type -> T4 GPU` only requests a GPU, it doesn't guarantee Colab attaches one (free-tier availability varies). `nvidia-smi` asks the OS directly, which is a more reliable check than `torch.cuda.is_available()` later, so it's worth confirming here first. If this errors with "command not found" or shows no GPU, stop and fix the runtime type (and reconnect) before continuing - `fish_extractor` will still run without a GPU, but SAM 2 segmentation across ~1,460 images on CPU is impractically slow.

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone (or update) the project on Drive

The first clone pulls ~1.8GB (the Phase 0 image archives are checked into the repo) and can take several minutes over Drive's write throughput - that's expected, let it run. Every session after the first just pulls code changes, which is fast.

Heads-up on Drive space: unzipping the archives (step 5 below) roughly doubles `data/raw_images/`'s footprint, and the extracted output adds more on top - budget **~6-8GB free** on the Drive account you mount here.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")

if not PROJECT_DIR.exists():
    print(f"Cloning into {PROJECT_DIR} (first time - pulls ~1.8GB, be patient)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"{PROJECT_DIR} already exists - pulling latest code only.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

In [ ]:
%cd {PROJECT_DIR}

## 3. Install dependencies

Colab already ships a CUDA-compatible `torch`/`torchvision`, so only the extra GPU packages are installed here (not the project's full `vision` extra, which would also try to reinstall torch). The local package isn't `pip install`ed - `src/` is added directly to `sys.path` instead, the same way the test suite imports it (see `pyproject.toml`'s `pythonpath`).

In [ ]:
%pip install -q "transformers>=4.51" accelerate opencv-python sam2

import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import torch
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("WARNING: no GPU detected - check Runtime > Change runtime type.")

## 4. (Optional but recommended) Smoke-test on one species first

Loading Grounding DINO + SAM 2 for the first time takes a few minutes and a couple GB of Hugging Face downloads. Worth confirming this works end-to-end on a single, fast species before committing to the full 64-species / ~1,460-image run.

In [ ]:
from fish_extractor.config import PipelineConfig
from fish_extractor.pipeline import FishExtractorPipeline

# Every path in PipelineConfig() is relative (data/..., reports/...) and
# resolves against PROJECT_DIR because of the %cd above - this is what keeps
# the review page's image links valid later (see review.py's relative-path
# handling), so don't override these to absolute paths.
config = PipelineConfig()

smoke_test = FishExtractorPipeline(config).run(species_filter={"Paracanthurus hepatus"})
accepted = [r for r in smoke_test if r.status == "accepted"]
flagged = [r for r in smoke_test if r.status == "flagged"]
print(f"Smoke test: {len(accepted)}/{len(smoke_test)} accepted, {len(flagged)} flagged.")

## 5. Full run (all 64 species)

Resumable: safe to re-run after a disconnect at any point. `run()` only returns images processed *this* call - anything already `accepted`/`flagged`/`excluded` from a prior call (including the smoke test above) is skipped, and state is saved after every single image, not just at the end.

In [ ]:
pipeline = FishExtractorPipeline(config)
results = pipeline.run()  # no species_filter = every species not already processed

accepted = [r for r in results if r.status == "accepted"]
flagged = [r for r in results if r.status == "flagged"]
print(f"This run: {len(accepted)}/{len(results)} accepted, {len(flagged)} flagged for review.")

## 6. Generate the review page

Anything the QA gate couldn't auto-accept (wrong number of fish, off-center, wrong size) needs a human decision. This writes a self-contained HTML page under Drive.

In [ ]:
from fish_extractor.review import generate_review_html

review_path = generate_review_html(config, config.review_path)
print(f"Review page written to: {review_path.resolve()}")

## 6b. View the review page

The image links in that page are *relative* (`../data/raw_images/...`), which only resolve correctly if something actually serves the whole project tree together - opening the `.html` file through Drive's own web preview (clicking it on drive.google.com) does **not** do this and shows blank/broken images, since Drive's preview sandboxes the file instead of serving its sibling folders.

The reliable way to view it without leaving Colab: start a tiny local web server rooted at the project folder and open it through Colab's authenticated port proxy, which serves the real directory tree so the relative links work. Leave this cell's server running while you review - stop it later with `review_server.terminate()` if you want the port back.

In [ ]:
import subprocess
import time

from google.colab import output

_REVIEW_PORT = 8000
review_server = subprocess.Popen(
    ["python3", "-m", "http.server", str(_REVIEW_PORT)], cwd=str(PROJECT_DIR)
)
time.sleep(1)  # give the server a moment to bind before opening the proxy
output.serve_kernel_port_as_window(_REVIEW_PORT, path=f"/{config.review_path.as_posix()}")

Check any images to exclude permanently, click **Export exclusions** - that downloads `fish_extraction_review_feedback.json` to your computer's normal Downloads folder (this is a real browser tab via the port proxy, so downloads behave normally). Upload/move that file into `reports/` on Drive (drag it into the Drive folder, or use Colab's file browser's upload button targeting that path), then run the next cell.

(If you'd rather review from your own machine outside Colab: this works fine too *if* Google Drive for Desktop has `My Drive` synced to a real local folder - then double-clicking `reports/fish_extraction_review.html` in your file explorer opens it via a real `file://` path with working sibling-folder links. Drive's web preview specifically is what breaks it, not Drive in general.)

## 7. Apply review feedback (after you've reviewed)

Run this only after exporting `fish_extraction_review_feedback.json` from the review page above and placing it in `reports/`. This permanently excludes the checked images; anything left unchecked stays flagged for a future pass, rather than being silently force-accepted or dropped.

In [ ]:
from fish_extractor.review import apply_review_feedback

feedback_path = config.review_path.parent / "fish_extraction_review_feedback.json"
if feedback_path.exists():
    excluded = apply_review_feedback(config, feedback_path)
    print(f"Applied review feedback: {excluded} image(s) marked permanently excluded.")
else:
    print(f"No feedback file found at {feedback_path} yet - complete the review step first.")

## 8. Status check

Re-run this any time to see progress without processing anything new. Once every image is `accepted`, `flagged`-and-resolved, or `excluded`, Phase 1 is done - bump the version per [CLAUDE.md](../CLAUDE.md) and move to `Phase2_Pattern_Extraction.ipynb`.

In [ ]:
import json
from collections import Counter

state = json.loads(config.state_path.read_text())
counts = Counter(v["status"] for v in state.values())
print(f"{sum(counts.values())} image(s) tracked: {dict(counts)}")